# BirdNET baseline — TFLite logits features (BirdCLEF 2026)

Experimental notebook: **BirdNET GLOBAL TFLite features (~6522-D logits per chunk) + small classification heads**, trained on focal `train_audio` and validated on labeled `train_soundscapes` windows (~739 rows).

**Why this path:** The `.tflite` shipped with `birdnetlib` exposes **species logits only**, not the internal 1024-D embedding (that needs the BirdNET SavedModel / `birdnet-analyzer` encode API). Logits are still a strong fixed representation for linear / small MLP heads. Bird-specific pretrained features via TFLite fit well on Apple Silicon. Keep this notebook **separate from `src/agent.py`** until you have a headline score.

**Stages (run top to bottom):**

1. Install `birdnetlib` + scientific stack
2. Paths, species order, multi-label soundscape targets
3. BirdNET feature helper (**TensorFlow Lite** + model path from `birdnetlib`)
4. **Stage 2 a)** Clean cache path: focal + labeled soundscape **feature vectors** → `.npz`
5. **Stage 2 b)** Mix cache path: focal-with-mixing + labeled soundscape **feature vectors** → `.npz`
6. Heads **A** (logistic OvR), **B** (weighted BCE MLP), **C** (same MLP + mixup on features)
7. **Ensemble:** mean of validation probabilities

**Window length:** `birdnetlib` feeds BirdNET at **48 kHz** with **`sample_secs = 3.0`** internally (see `RecordingBase`). We load up to **5 s** from disk at **32 kHz** (competition familiarity), resample to **48 kHz**, then **center-crop / pad to 3 s** so the bundled TFLite model sees the expected chunk length.


## Stage 1 — Install

Run once per environment. BirdNET ships as **TFLite** inside `birdnetlib`; TensorFlow provides the interpreter.

In [1]:
%pip install -q birdnetlib librosa scipy scikit-learn tensorflow tqdm

Note: you may need to restart the kernel to use updated packages.


## Imports, paths, constants

In [2]:
from __future__ import annotations

import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.multioutput import MultiOutputClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

_HERE = Path.cwd().resolve()
PROJECT_ROOT = _HERE.parent if _HERE.name == "notebooks" else _HERE

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_AUDIO_DIR = DATA_DIR / "train_audio"
TRAIN_SOUNDSCAPES_DIR = DATA_DIR / "train_soundscapes"
TRAIN_CSV = DATA_DIR / "train.csv"
LABELS_CSV = DATA_DIR / "train_soundscapes_labels.csv"
SAMPLE_SUB_CSV = DATA_DIR / "sample_submission.csv"

CACHE_DIR = PROJECT_ROOT / "notebooks" / "birdnet_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NPZ_CLEAN = CACHE_DIR / "train_embeddings_clean.npz"
NPZ_MIX = CACHE_DIR / "train_embeddings_mix.npz"
NPZ_VAL = CACHE_DIR / "soundscape_val_embeddings.npz"

SR_LOAD = 32000
BIRDNET_SR = 48000
CLIP_LOAD_SEC = 5.0
BIRDNET_CHUNK_SEC = 3.0
# Single TFLite output = BirdNET GLOBAL 6K species logits (~6522 floats), not penultimate 1024-D embedding.
EMB_DIM = 6522

MIX_PROB = 0.35
SNR_MIN_DB, SNR_MAX_DB = 0.0, 15.0
N_MIX_VIEWS = 3
RNG = np.random.default_rng(42)

print("PROJECT_ROOT:", PROJECT_ROOT)

/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PROJECT_ROOT: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project


/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Species order + labeled soundscape targets

In [3]:
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
species_cols = [c for c in sample_sub.columns if c != "row_id"]
species_to_idx = {s: i for i, s in enumerate(species_cols)}
n_species = len(species_cols)
print("n_species:", n_species)

train_df = pd.read_csv(TRAIN_CSV)


def _tok(v) -> set[str]:
    if pd.isna(v) or v == "":
        return set()
    return {t.strip() for t in str(v).split(";") if t.strip()}


def _merge_labels(series: pd.Series) -> set[str]:
    o: set[str] = set()
    for v in series:
        o |= _tok(v)
    return o


lab = pd.read_csv(LABELS_CSV)
grp = (
    lab.groupby(["filename", "start", "end"], sort=False)["primary_label"]
    .agg(_merge_labels)
    .reset_index()
)
grp["end_sec"] = pd.to_timedelta(grp["end"]).dt.total_seconds().astype(int)
grp["row_id"] = grp["filename"].str.replace(".ogg", "", regex=False) + "_" + grp["end_sec"].astype(str)

y_true_rows = []
unknown_codes: set[str] = set()
for _, r in grp.iterrows():
    v = np.zeros(n_species, dtype=np.float32)
    for code in r["primary_label"]:
        j = species_to_idx.get(code)
        if j is None:
            unknown_codes.add(code)
        else:
            v[j] = 1.0
    y_true_rows.append((r["row_id"], v))

if unknown_codes:
    print("WARNING: label tokens missing from sample_submission:", len(unknown_codes))

y_true_df = pd.DataFrame(
    [v for _, v in y_true_rows], index=[rid for rid, _ in y_true_rows], columns=species_cols
).sort_index()
if not y_true_df.index.is_unique:
    y_true_df = y_true_df.groupby(level=0).max()

labeled_stems = {rid.rsplit("_", 1)[0] for rid in y_true_df.index}
print("Labeled soundscape windows:", len(y_true_df))

n_species: 234
Labeled soundscape windows: 739


## Stage 1 — BirdNET feature helper

We call **TensorFlow Lite directly** on the bundled BirdNET `.tflite` (path from `birdnetlib.analyzer.MODEL_PATH`). This avoids a macOS/Jupyter bug where `RecordingBuffer.extract_embeddings()` can hit `ValueError: Tensor data is null` after repeated interpreter resizes.

**Important:** This export has **one** output tensor: **~6522 logits**. TensorFlow Lite does not populate the older `output_index - 1` “embedding” slot birdnetlib uses; that path returns the wrong buffer or null. For true **1024-D** embeddings, use the official BirdNET SavedModel / `birdnet-analyzer` encoding API instead of this notebook’s TFLite file.

Audio: resample/pad to one **3 s** chunk at **48 kHz**, then flatten the **logit vector** as the feature row.

In [5]:
import threading

from birdnetlib.analyzer import MODEL_PATH

_EMB_LOCK = threading.Lock()
_tflite = None
_tflite_in_idx = None
_tflite_out_idx = None


def _init_birdnet_tflite() -> None:
    global _tflite, _tflite_in_idx, _tflite_out_idx
    if _tflite is not None:
        return
    intr = tf.lite.Interpreter(model_path=str(MODEL_PATH), num_threads=1)
    intr.allocate_tensors()
    inp = intr.get_input_details()[0]
    _tflite_in_idx = int(inp["index"])
    outs = intr.get_output_details()
    _tflite_out_idx = int(outs[0]["index"])
    odim = int(np.asarray(outs[0]["shape"])[-1])
    _tflite = intr
    print("BirdNET TFLite:", MODEL_PATH, "| output_index=", _tflite_out_idx, "| feature_dim=", odim)
    if odim != EMB_DIM:
        print(
            "WARNING: set EMB_DIM in the constants cell to",
            odim,
            "to match this model (or switch to a different .tflite).",
        )


def audio_to_birdnet_chunk(y: np.ndarray, sr: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if sr != BIRDNET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=BIRDNET_SR)
    max_keep = int(CLIP_LOAD_SEC * BIRDNET_SR)
    if len(y) > max_keep:
        y = y[:max_keep]
    need = int(BIRDNET_CHUNK_SEC * BIRDNET_SR)
    if len(y) >= need:
        s0 = max(0, (len(y) - need) // 2)
        y = y[s0 : s0 + need]
    else:
        y = np.pad(y, (0, need - len(y)))
    return y.astype(np.float32)


def embed_audio_mono(y: np.ndarray, sr: int, *, analyzer=None) -> np.ndarray:
    _ = analyzer
    chunk = audio_to_birdnet_chunk(y, sr)
    chunk = np.ascontiguousarray(chunk, dtype=np.float32).reshape(1, -1)
    with _EMB_LOCK:
        _init_birdnet_tflite()
        intr = _tflite
        assert intr is not None and _tflite_in_idx is not None and _tflite_out_idx is not None
        intr.resize_tensor_input(_tflite_in_idx, chunk.shape)
        intr.allocate_tensors()
        intr.set_tensor(_tflite_in_idx, chunk)
        intr.invoke()
        feat = intr.get_tensor(_tflite_out_idx)
    v = np.asarray(feat, dtype=np.float32).reshape(-1)
    if v.shape[0] != EMB_DIM:
        raise RuntimeError(f"Unexpected BirdNET output dim {v.shape}; expected EMB_DIM={EMB_DIM}")
    return v


def load_focal_clip(path: Path) -> tuple[np.ndarray, int]:
    y, sr = librosa.load(str(path), sr=SR_LOAD, mono=True)
    n = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(y) > n:
        y = y[:n]
    elif len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD

## Stage 2 a) — Cache embeddings (clean focal + labeled windows)

- Independent path A: run this for clean focal training cache + validation cache.
- Set `FORCE_REBUILD = True` to overwrite caches.
- Set `MAX_FOCAL = 2000` (for example) for a fast smoke test.
- Skips missing audio paths silently.

In [ ]:
FORCE_REBUILD = False
MAX_FOCAL: int | None = None


def macro_auc_ge3(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, int, int]:
    pos = y_true.sum(axis=0)
    keep = pos >= 3
    if not np.any(keep):
        return float("nan"), 0, int(np.sum(pos > 0))
    yt = y_true[:, keep]
    ys = y_score[:, keep]
    usable = []
    for j in range(yt.shape[1]):
        col = yt[:, j]
        if col.min() == 0 and col.max() == 1:
            usable.append(j)
    if not usable:
        return float("nan"), int(np.sum(keep)), int(np.sum(pos > 0))
    usable = np.array(usable, dtype=int)
    auc = roc_auc_score(yt[:, usable], ys[:, usable], average="macro")
    return float(auc), int(np.sum(keep)), int(np.sum(pos > 0))


def build_focal_manifest() -> pd.DataFrame:
    rows = []
    for _, r in train_df.iterrows():
        lab = str(r["primary_label"])
        rel = r["filename"]
        if lab not in species_to_idx:
            continue
        ap = TRAIN_AUDIO_DIR / str(rel)
        if not ap.is_file():
            continue
        rows.append({"path": str(ap.resolve()), "primary_label": lab})
    mf = pd.DataFrame(rows)
    if MAX_FOCAL is not None:
        mf = mf.head(int(MAX_FOCAL))
    return mf


def soundscape_window_audio(row_id: str) -> tuple[np.ndarray, int]:
    stem, end_s = row_id.rsplit("_", 1)
    end_sec = int(end_s)
    start_sec = max(0, end_sec - int(CLIP_LOAD_SEC))
    fp = TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg"
    y, _ = librosa.load(
        str(fp), sr=SR_LOAD, mono=True, offset=start_sec, duration=CLIP_LOAD_SEC
    )
    n = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(y) > n:
        y = y[:n]
    elif len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD


if FORCE_REBUILD or not NPZ_CLEAN.is_file() or not NPZ_VAL.is_file():
    manifest = build_focal_manifest()
    print("Focal rows to embed:", len(manifest))

    X_list, y_list, p_list = [], [], []
    for r in tqdm(manifest.itertuples(index=False), total=len(manifest)):
        y_wav, sr = load_focal_clip(Path(r.path))
        emb = embed_audio_mono(y_wav, sr)
        vec = np.zeros(n_species, dtype=np.float32)
        vec[species_to_idx[r.primary_label]] = 1.0
        X_list.append(emb)
        y_list.append(vec)
        p_list.append(r.path)

    X_tr = np.stack(X_list, axis=0)
    y_tr = np.stack(y_list, axis=0)

    X_va, rid_va = [], []
    for rid in tqdm(list(y_true_df.index)):
        stem = rid.rsplit("_", 1)[0]
        if not (TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg").is_file():
            continue
        y_wav, sr = soundscape_window_audio(rid)
        X_va.append(embed_audio_mono(y_wav, sr))
        rid_va.append(rid)

    X_va = np.stack(X_va, axis=0)
    rid_va = np.array(rid_va, dtype=object)
    y_va = y_true_df.loc[rid_va].to_numpy(dtype=np.float32)

    np.savez_compressed(NPZ_CLEAN, X_train=X_tr, y_train=y_tr, train_paths=np.array(p_list, dtype=object))
    np.savez_compressed(NPZ_VAL, X_val=X_va, row_ids=rid_va, y_val=y_va)
    print("Saved", NPZ_CLEAN, X_tr.shape)
    print("Saved", NPZ_VAL, X_va.shape)
else:
    d1 = np.load(NPZ_CLEAN, allow_pickle=True)
    d2 = np.load(NPZ_VAL, allow_pickle=True)
    print("Loaded", NPZ_CLEAN, d1["X_train"].shape, "|", NPZ_VAL, d2["X_val"].shape)

## Stage 2 b) — Mix focal cache + validation cache

Builds mixed focal training features (for `USE_MIX_TRAIN=True`) by sampling backgrounds from **`train_soundscapes` files whose stems are not in the labeled evaluation set** and mixing at random SNR.

Also ensures `soundscape_val_embeddings.npz` exists, so this stage can be run independently of Stage 2 a) for the mix-training path.

Cost: training cache is roughly **`× N_MIX_VIEWS`** inference work versus clean focal caching.

In [13]:
FORCE_REBUILD_MIX = False
FORCE_REBUILD_VAL_FROM_STAGE4 = False

noise_candidates = sorted(TRAIN_SOUNDSCAPES_DIR.glob("*.ogg"))
noise_pool = [p for p in noise_candidates if p.stem not in labeled_stems]
print("Noise pool files:", len(noise_pool))


def random_soundscape_noise_5s() -> np.ndarray:
    if not noise_pool:
        raise RuntimeError("No soundscapes available for mixing.")
    fp = Path(noise_pool[int(RNG.integers(0, len(noise_pool)))])
    dur = librosa.get_duration(path=str(fp))
    start_max = max(0.0, dur - CLIP_LOAD_SEC)
    offset = float(RNG.uniform(0, start_max)) if start_max > 0 else 0.0
    n, _sr = librosa.load(str(fp), sr=SR_LOAD, mono=True, offset=offset, duration=CLIP_LOAD_SEC)
    nt = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(n) > nt:
        n = n[:nt]
    elif len(n) < nt:
        n = np.pad(n, (0, nt - len(n)))
    return n.astype(np.float32)


def mix_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    s = signal.astype(np.float32)
    n = noise.astype(np.float32)
    ps = np.mean(s ** 2) + 1e-12
    pn = np.mean(n ** 2) + 1e-12
    scale = np.sqrt(ps / (pn * (10 ** (snr_db / 10.0))))
    return np.clip(s + scale * n, -1.0, 1.0)


def build_focal_manifest_stage4() -> pd.DataFrame:
    rows = []
    for _, r in train_df.iterrows():
        lab = str(r["primary_label"])
        rel = r["filename"]
        if lab not in species_to_idx:
            continue
        ap = TRAIN_AUDIO_DIR / str(rel)
        if not ap.is_file():
            continue
        rows.append({"path": str(ap.resolve()), "primary_label": lab})
    return pd.DataFrame(rows)


def soundscape_window_audio_stage4(row_id: str) -> tuple[np.ndarray, int]:
    stem, end_s = row_id.rsplit("_", 1)
    end_sec = int(end_s)
    start_sec = max(0, end_sec - int(CLIP_LOAD_SEC))
    fp = TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg"
    y, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True, offset=start_sec, duration=CLIP_LOAD_SEC)
    n = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(y) > n:
        y = y[:n]
    elif len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD


if FORCE_REBUILD_MIX or not NPZ_MIX.is_file():
    manifest = build_focal_manifest_stage4()
    Xmx, ymx, meta_path, meta_vid = [], [], [], []

    for r in tqdm(manifest.itertuples(index=False), total=len(manifest), desc="Stage4 mix"):
        y_clean, sr = load_focal_clip(Path(r.path))
        lbl_vec = np.zeros(n_species, dtype=np.float32)
        lbl_vec[species_to_idx[r.primary_label]] = 1.0

        for v in range(N_MIX_VIEWS):
            if v > 0 and noise_pool and RNG.random() < MIX_PROB:
                noise = random_soundscape_noise_5s()
                snr = float(RNG.uniform(SNR_MIN_DB, SNR_MAX_DB))
                y_use = mix_snr(y_clean, noise, snr)
            else:
                y_use = y_clean
            emb = embed_audio_mono(y_use, sr)
            Xmx.append(emb)
            ymx.append(lbl_vec.copy())
            meta_path.append(r.path)
            meta_vid.append(np.int16(v))

    np.savez_compressed(
        NPZ_MIX,
        X_train=np.stack(Xmx),
        y_train=np.stack(ymx),
        paths=np.array(meta_path, dtype=object),
        variant=np.array(meta_vid),
    )
    print("Saved", NPZ_MIX, np.stack(Xmx).shape)
else:
    print("Mix cache exists:", NPZ_MIX)


if FORCE_REBUILD_VAL_FROM_STAGE4 or not NPZ_VAL.is_file():
    X_va, rid_va = [], []
    for rid in tqdm(list(y_true_df.index), desc="Stage4 val cache"):
        stem = rid.rsplit("_", 1)[0]
        if not (TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg").is_file():
            continue
        y_wav, sr = soundscape_window_audio_stage4(rid)
        X_va.append(embed_audio_mono(y_wav, sr))
        rid_va.append(rid)

    X_va = np.stack(X_va, axis=0)
    rid_va = np.array(rid_va, dtype=object)
    y_va = y_true_df.loc[rid_va].to_numpy(dtype=np.float32)
    np.savez_compressed(NPZ_VAL, X_val=X_va, row_ids=rid_va, y_val=y_va)
    print("Saved", NPZ_VAL, X_va.shape)
else:
    d2 = np.load(NPZ_VAL, allow_pickle=True)
    print("Val cache exists:", NPZ_VAL, d2["X_val"].shape)

Noise pool files: 10592


Stage4 mix:   0%|          | 2/35549 [00:00<3:57:19,  2.50it/s]

BirdNET TFLite: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv/lib/python3.9/site-packages/birdnetlib/models/analyzer/BirdNET_GLOBAL_6K_V2.4_Model_FP32.tflite | output_index= 546 | feature_dim= 6522


Stage4 mix: 100%|██████████| 35549/35549 [1:05:38<00:00,  9.03it/s]


Saved /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/birdnet_cache/train_embeddings_mix.npz (106647, 6522)


Stage4 val cache: 100%|██████████| 739/739 [00:24<00:00, 30.59it/s]


Saved /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/birdnet_cache/soundscape_val_embeddings.npz (739, 6522)


## Stage 3 — Heads A, B, C

- Toggle **`USE_MIX_TRAIN`** to train on mixed embedding cache.
- Validation always uses **`soundscape_val_embeddings.npz`** (clean BirdNET windows).
- Metric: **macro ROC-AUC** over species with **≥3 positives** in the aligned validation matrix (same spirit as your validation notebook).

In [10]:
USE_MIX_TRAIN = True

import json
import pickle

AUTOSAVE_ROOT = PROJECT_ROOT / "notebooks" / "submission_birdnet"
AUTOSAVE_RUN_NAME = "submission_augmented_birdnet"
AUTOSAVE_DIR = AUTOSAVE_ROOT / AUTOSAVE_RUN_NAME
AUTOSAVE_DIR.mkdir(parents=True, exist_ok=True)
AUTOSAVE_METRICS_PATH = AUTOSAVE_DIR / "metrics_stage3.json"

def _save_stage3_metrics(**kwargs):
    payload = {}
    if AUTOSAVE_METRICS_PATH.is_file():
        with open(AUTOSAVE_METRICS_PATH, "r", encoding="utf-8") as f:
            payload = json.load(f)
    payload.update(kwargs)
    with open(AUTOSAVE_METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def macro_auc_ge3(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, int, int]:
    pos = y_true.sum(axis=0)
    keep = pos >= 3
    if not np.any(keep):
        return float("nan"), 0, int(np.sum(pos > 0))
    yt = y_true[:, keep]
    ys = y_score[:, keep]
    usable = []
    for j in range(yt.shape[1]):
        col = yt[:, j]
        if col.min() == 0 and col.max() == 1:
            usable.append(j)
    if not usable:
        return float("nan"), int(np.sum(keep)), int(np.sum(pos > 0))
    usable = np.array(usable, dtype=int)
    auc = roc_auc_score(yt[:, usable], ys[:, usable], average="macro")
    return float(auc), int(np.sum(keep)), int(np.sum(pos > 0))


dval = np.load(NPZ_VAL, allow_pickle=True)
X_val, y_val = dval["X_val"].astype(np.float32), dval["y_val"].astype(np.float32)

train_npz = NPZ_MIX if USE_MIX_TRAIN and NPZ_MIX.is_file() else NPZ_CLEAN
dtr = np.load(train_npz, allow_pickle=True)
X_train, y_train = dtr["X_train"].astype(np.float32), dtr["y_train"].astype(np.float32)
print("Train:", train_npz.name, X_train.shape, "| Val:", X_val.shape)
_save_stage3_metrics(train_npz=str(train_npz), x_train_shape=list(X_train.shape), x_val_shape=list(X_val.shape))

Train: train_embeddings_mix.npz (106647, 6522) | Val: (739, 6522)


In [ ]:
# --- Head A: OvR logistic regression ---
# Some species may be absent in the current training cache (all-zero column),
# which makes binary logistic regression fail. Train only valid columns and
# then scatter back into the full species matrix.
valid_cols_a = np.where((y_train.min(axis=0) < y_train.max(axis=0)))[0]
logreg_valid_cols = valid_cols_a.astype(np.int32)
pred_a = np.zeros((len(X_val), n_species), dtype=np.float32)
if len(valid_cols_a) == 0:
    print("Head A skipped: no trainable columns with both classes.")
    auc_a, k3_a, _ = macro_auc_ge3(y_val, pred_a)
else:
    logreg = MultiOutputClassifier(
        LogisticRegression(max_iter=300, class_weight="balanced", solver="lbfgs", n_jobs=1),
        n_jobs=1,
    )
    logreg.fit(X_train, y_train[:, valid_cols_a])
    pred_a_valid = np.stack([est.predict_proba(X_val)[:, 1] for est in logreg.estimators_], axis=1)
    pred_a[:, valid_cols_a] = pred_a_valid.astype(np.float32)
    auc_a, k3_a, _ = macro_auc_ge3(y_val, pred_a)
print(f"Head A macro-AUC (≥3 pos): {auc_a:.5f} | species_kept≥3: {k3_a} | trained_cols: {len(valid_cols_a)}")

head_a_path = AUTOSAVE_DIR / "head_a_logreg.pkl"
if len(valid_cols_a) > 0:
    with open(head_a_path, "wb") as f:
        pickle.dump(logreg, f)
    head_a_path_str = str(head_a_path)
    print("Saved Head A:", head_a_path)
else:
    head_a_path_str = ""
    print("Head A model not saved (no trainable columns).")
_save_stage3_metrics(
    auc_a=float(auc_a),
    k3_a=int(k3_a),
    logreg_valid_cols=[int(x) for x in logreg_valid_cols.tolist()],
    head_a_path=head_a_path_str,
)

# Head B and Head C are split into separate cells below so they can be run independently.
# This cell intentionally stops after Head A.

Train: train_embeddings_mix.npz (106647, 6522) | Val: (739, 6522)


: 

In [11]:
# --- Head B: MLP + per-class positive weights (clip 1..25) ---
pos = y_train.sum(axis=0).astype(np.float64)
neg = len(y_train) - pos
pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
pw_v = tf.constant(pos_weight)[tf.newaxis, :]


def weighted_multilabel_bce(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    return tf.reduce_mean(
        pw_v * y_true * (-tf.math.log(y_pred))
        + (1.0 - y_true) * (-tf.math.log(1.0 - y_pred)),
        axis=-1,
    )


tf.keras.utils.set_random_seed(42)
inp = tf.keras.layers.Input(shape=(EMB_DIM,))
h = tf.keras.layers.Dense(512, activation="relu")(inp)
h = tf.keras.layers.Dropout(0.3)(h)
out = tf.keras.layers.Dense(n_species, activation="sigmoid")(h)
mlp_b = tf.keras.Model(inp, out)
mlp_b.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=weighted_multilabel_bce)
cb = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss")
mlp_b.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=256,
    callbacks=[cb],
    verbose=1,
)
pred_b = mlp_b.predict(X_val, batch_size=512, verbose=0)
auc_b, k3_b, _ = macro_auc_ge3(y_val, pred_b)
print(f"Head B macro-AUC (≥3 pos): {auc_b:.5f} | species_kept≥3: {k3_b}")

head_b_path = AUTOSAVE_DIR / "head_b_mlp.keras"
mlp_b.save(head_b_path)
_save_stage3_metrics(auc_b=float(auc_b), k3_b=int(k3_b), head_b_path=str(head_b_path))
print("Saved Head B:", head_b_path)

Epoch 1/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 2.0258 - val_loss: 1.7220
Epoch 2/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 1.7221 - val_loss: 1.7220
Epoch 3/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 1.7221 - val_loss: 1.7220
Epoch 4/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 1.7221 - val_loss: 1.7220
Epoch 5/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 1.7220 - val_loss: 1.7220
Epoch 6/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 1.7220 - val_loss: 1.7220
Head B macro-AUC (≥3 pos): 0.50000 | species_kept≥3: 64
Saved Head B: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/submission_birdnet/submission_augmented_birdnet/head_b_mlp.keras


## Stage 3b (optional) — Normalized feature retry for B/C

If Heads B/C collapse to ~0.5 AUC, run this step to standardize BirdNET features (z-score per dimension) and retrain B/C with plain BCE.

Artifacts are saved separately so you can compare against the original heads without overwriting them.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1) Standardize BirdNET feature dimensions using train split stats.
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_z = scaler.fit_transform(X_train).astype(np.float32)
X_val_z = scaler.transform(X_val).astype(np.float32)

scaler_path = AUTOSAVE_DIR / "feature_scaler_stage3.npz"
np.savez_compressed(
    scaler_path,
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32),
)
print("Saved scaler:", scaler_path)

# 2) Head B (normalized features, plain BCE).
tf.keras.utils.set_random_seed(42)
inp_bz = tf.keras.layers.Input(shape=(EMB_DIM,))
h_bz = tf.keras.layers.Dense(512, activation="relu")(inp_bz)
h_bz = tf.keras.layers.Dropout(0.3)(h_bz)
out_bz = tf.keras.layers.Dense(n_species, activation="sigmoid")(h_bz)
mlp_b_norm = tf.keras.Model(inp_bz, out_bz)
mlp_b_norm.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
cb_bz = tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_loss")
mlp_b_norm.fit(
    X_train_z,
    y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=256,
    callbacks=[cb_bz],
    verbose=1,
)
pred_b_norm = mlp_b_norm.predict(X_val_z, batch_size=512, verbose=0)
auc_b_norm, k3_b_norm, _ = macro_auc_ge3(y_val, pred_b_norm)
print(f"Head B (norm) macro-AUC (≥3 pos): {auc_b_norm:.5f} | species_kept≥3: {k3_b_norm}")

head_b_norm_path = AUTOSAVE_DIR / "head_b_mlp_norm.keras"
mlp_b_norm.save(head_b_norm_path)
print("Saved Head B norm:", head_b_norm_path)

# 3) Head C (normalized features + mixup, plain BCE).
def train_mlp_mixup_numpy_plain(model, X, y, epochs=80, batch_size=256, lr=1e-3, alpha=0.4):
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    n = len(X)
    idx = np.arange(n)

    @tf.function
    def train_step(xb, yb):
        with tf.GradientTape() as tape:
            preds = model(xb, training=True)
            loss = tf.reduce_mean(tf.reduce_mean(tf.keras.losses.binary_crossentropy(yb, preds), axis=-1))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for ep in range(epochs):
        RNG.shuffle(idx)
        losses = []
        for start in range(0, n, batch_size):
            sel = idx[start : start + batch_size]
            xb = X[sel].copy()
            yb = y[sel].copy()
            if len(xb) > 1:
                lam = RNG.beta(alpha, alpha, size=(len(xb), 1)).astype(np.float32)
                lam = np.maximum(lam, 1.0 - lam)
                perm = RNG.permutation(len(xb))
                xb = lam * xb + (1.0 - lam) * xb[perm]
                yb = lam * yb + (1.0 - lam) * yb[perm]
            loss = train_step(tf.constant(xb), tf.constant(yb))
            losses.append(float(loss.numpy()))
        if (ep + 1) % 10 == 0:
            print(f"  mixup(norm) epoch {ep+1} loss={np.mean(losses):.5f}")


inp_cz = tf.keras.layers.Input(shape=(EMB_DIM,))
h_cz = tf.keras.layers.Dense(512, activation="relu")(inp_cz)
h_cz = tf.keras.layers.Dropout(0.3)(h_cz)
out_cz = tf.keras.layers.Dense(n_species, activation="sigmoid")(h_cz)
mlp_c_norm = tf.keras.Model(inp_cz, out_cz)
mlp_c_norm.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
train_mlp_mixup_numpy_plain(mlp_c_norm, X_train_z, y_train, epochs=80, batch_size=256)
pred_c_norm = mlp_c_norm.predict(X_val_z, batch_size=512, verbose=0)
auc_c_norm, k3_c_norm, _ = macro_auc_ge3(y_val, pred_c_norm)
print(f"Head C (norm) macro-AUC (≥3 pos): {auc_c_norm:.5f} | species_kept≥3: {k3_c_norm}")

head_c_norm_path = AUTOSAVE_DIR / "head_c_mlp_mixup_norm.keras"
mlp_c_norm.save(head_c_norm_path)
print("Saved Head C norm:", head_c_norm_path)

_save_stage3_metrics(
    scaler_path=str(scaler_path),
    auc_b_norm=float(auc_b_norm),
    k3_b_norm=int(k3_b_norm),
    head_b_norm_path=str(head_b_norm_path),
    auc_c_norm=float(auc_c_norm),
    k3_c_norm=int(k3_c_norm),
    head_c_norm_path=str(head_c_norm_path),
)


In [12]:
# --- Head C: same architecture, train with feature-space mixup (Beta 0.4) ---
# Requires weighted_multilabel_bce from the Head B cell.

def train_mlp_mixup_numpy(model, X, y, epochs=80, batch_size=256, lr=1e-3, alpha=0.4):
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    n = len(X)
    idx = np.arange(n)

    @tf.function
    def train_step(xb, yb):
        with tf.GradientTape() as tape:
            preds = model(xb, training=True)
            loss = tf.reduce_mean(weighted_multilabel_bce(yb, preds))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for ep in range(epochs):
        RNG.shuffle(idx)
        losses = []
        for start in range(0, n, batch_size):
            sel = idx[start : start + batch_size]
            xb = X[sel].copy()
            yb = y[sel].copy()
            if len(xb) > 1:
                lam = RNG.beta(alpha, alpha, size=(len(xb), 1)).astype(np.float32)
                lam = np.maximum(lam, 1.0 - lam)
                perm = RNG.permutation(len(xb))
                xb = lam * xb + (1.0 - lam) * xb[perm]
                yb = lam * yb + (1.0 - lam) * yb[perm]
            loss = train_step(tf.constant(xb), tf.constant(yb))
            losses.append(float(loss.numpy()))
        if (ep + 1) % 10 == 0:
            print(f"  mixup epoch {ep+1} loss={np.mean(losses):.5f}")


inp_c = tf.keras.layers.Input(shape=(EMB_DIM,))
h_c = tf.keras.layers.Dense(512, activation="relu")(inp_c)
h_c = tf.keras.layers.Dropout(0.3)(h_c)
out_c = tf.keras.layers.Dense(n_species, activation="sigmoid")(h_c)
mlp_c = tf.keras.Model(inp_c, out_c)
mlp_c.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=weighted_multilabel_bce)
train_mlp_mixup_numpy(mlp_c, X_train, y_train, epochs=80, batch_size=256)
pred_c = mlp_c.predict(X_val, batch_size=512, verbose=0)
auc_c, k3_c, _ = macro_auc_ge3(y_val, pred_c)
print(f"Head C macro-AUC (≥3 pos): {auc_c:.5f} | species_kept≥3: {k3_c}")

head_c_path = AUTOSAVE_DIR / "head_c_mlp_mixup.keras"
mlp_c.save(head_c_path)
_save_stage3_metrics(auc_c=float(auc_c), k3_c=int(k3_c), head_c_path=str(head_c_path))
print("Saved Head C:", head_c_path)

  mixup epoch 10 loss=1.72202
  mixup epoch 20 loss=1.72206
  mixup epoch 30 loss=1.72205
  mixup epoch 40 loss=1.72203
  mixup epoch 50 loss=1.72203
  mixup epoch 60 loss=1.72203
  mixup epoch 70 loss=1.72202
  mixup epoch 80 loss=1.72202
Head C macro-AUC (≥3 pos): 0.50000 | species_kept≥3: 64
Saved Head C: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/submission_birdnet/submission_augmented_birdnet/head_c_mlp_mixup.keras


## Stage 7 — Export submission bundle (mix-trained)

Saves a reusable artifact folder with:

- `head_a_logreg.pkl` (sklearn OvR model)
- `head_b_mlp.keras` (Keras)
- `head_c_mlp_mixup.keras` (Keras)
- `metadata.npz` (species order + audio constants)
- `kaggle_inference.ipynb` (BirdNET-feature inference notebook)

This stage is intended for your **augmented/mix run** (`USE_MIX_TRAIN=True`).

In [ ]:
import json
import pickle
from datetime import datetime

BUNDLE_ROOT = PROJECT_ROOT / "notebooks" / "submission_birdnet"
RUN_NAME = "submission_augmented_birdnet"
BUNDLE_DIR = BUNDLE_ROOT / RUN_NAME
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

if not USE_MIX_TRAIN:
    print("WARNING: USE_MIX_TRAIN is False. You are exporting models from a clean-training run.")

# Save trained heads.
pickle_path = BUNDLE_DIR / "head_a_logreg.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(logreg, f)

mlp_b_path = BUNDLE_DIR / "head_b_mlp.keras"
mlp_c_path = BUNDLE_DIR / "head_c_mlp_mixup.keras"
mlp_b.save(mlp_b_path)
mlp_c.save(mlp_c_path)

# Save metadata required for deterministic inference on Kaggle.
meta_path = BUNDLE_DIR / "metadata.npz"
np.savez_compressed(
    meta_path,
    species_cols=np.array(species_cols, dtype=object),
    logreg_valid_cols=logreg_valid_cols,
    emb_dim=np.int32(EMB_DIM),
    sr_load=np.int32(SR_LOAD),
    birdnet_sr=np.int32(BIRDNET_SR),
    clip_load_sec=np.float32(CLIP_LOAD_SEC),
    birdnet_chunk_sec=np.float32(BIRDNET_CHUNK_SEC),
)

# Build a Kaggle inference notebook that matches this BirdNET-feature pipeline.
kaggle_nb = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# BirdCLEF 2026 — BirdNET logits + head ensemble inference\\n",
                "\\n",
                "Upload this bundle folder as a Kaggle Dataset and set `MODEL_DIR` accordingly."
            ],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": None,
            "outputs": [],
            "source": [
                "%pip install -q birdnetlib librosa scikit-learn tensorflow"
            ],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": None,
            "outputs": [],
            "source": [
                "import os\\n",
                "import pickle\\n",
                "from pathlib import Path\\n",
                "\\n",
                "import librosa\\n",
                "import numpy as np\\n",
                "import pandas as pd\\n",
                "import tensorflow as tf\\n",
                "from birdnetlib.analyzer import MODEL_PATH\\n",
                "\\n",
                "COMP_DIR = '/kaggle/input/birdclef-2026'\\n",
                "MODEL_DIR = '/kaggle/input/REPLACE_WITH_YOUR_DATASET_FOLDER/submission_augmented_birdnet'\\n",
                "\\n",
                "meta = np.load(os.path.join(MODEL_DIR, 'metadata.npz'), allow_pickle=True)\\n",
                "species_cols = list(meta['species_cols'])\\n",
                "logreg_valid_cols = np.asarray(meta['logreg_valid_cols'], dtype=np.int32)\\n",
                "EMB_DIM = int(meta['emb_dim'])\\n",
                "SR_LOAD = int(meta['sr_load'])\\n",
                "BIRDNET_SR = int(meta['birdnet_sr'])\\n",
                "CLIP_LOAD_SEC = float(meta['clip_load_sec'])\\n",
                "BIRDNET_CHUNK_SEC = float(meta['birdnet_chunk_sec'])\\n",
                "\\n",
                "with open(os.path.join(MODEL_DIR, 'head_a_logreg.pkl'), 'rb') as f:\\n",
                "    logreg = pickle.load(f)\\n",
                "mlp_b = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'head_b_mlp.keras'), compile=False)\\n",
                "mlp_c = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'head_c_mlp_mixup.keras'), compile=False)\\n",
                "\\n",
                "intr = tf.lite.Interpreter(model_path=str(MODEL_PATH), num_threads=1)\\n",
                "intr.allocate_tensors()\\n",
                "in_idx = int(intr.get_input_details()[0]['index'])\\n",
                "out_idx = int(intr.get_output_details()[0]['index'])\\n",
                "\\n",
                "def audio_to_birdnet_chunk(y, sr):\\n",
                "    y = np.asarray(y, dtype=np.float32).reshape(-1)\\n",
                "    if sr != BIRDNET_SR:\\n",
                "        y = librosa.resample(y, orig_sr=sr, target_sr=BIRDNET_SR)\\n",
                "    max_keep = int(CLIP_LOAD_SEC * BIRDNET_SR)\\n",
                "    if len(y) > max_keep:\\n",
                "        y = y[:max_keep]\\n",
                "    need = int(BIRDNET_CHUNK_SEC * BIRDNET_SR)\\n",
                "    if len(y) >= need:\\n",
                "        s0 = max(0, (len(y) - need) // 2)\\n",
                "        y = y[s0:s0+need]\\n",
                "    else:\\n",
                "        y = np.pad(y, (0, need - len(y)))\\n",
                "    return y.astype(np.float32)\\n",
                "\\n",
                "def extract_feat(y, sr):\\n",
                "    chunk = audio_to_birdnet_chunk(y, sr).reshape(1, -1).astype(np.float32)\\n",
                "    intr.resize_tensor_input(in_idx, chunk.shape)\\n",
                "    intr.allocate_tensors()\\n",
                "    intr.set_tensor(in_idx, chunk)\\n",
                "    intr.invoke()\\n",
                "    feat = np.asarray(intr.get_tensor(out_idx), dtype=np.float32).reshape(-1)\\n",
                "    if feat.shape[0] != EMB_DIM:\\n",
                "        raise RuntimeError(f'Unexpected BirdNET output dim {feat.shape}; expected {EMB_DIM}')\\n",
                "    return feat\\n",
                "\\n",
                "def predict_heads(X):\\n",
                "    pa = np.zeros((len(X), len(species_cols)), dtype=np.float32)\\n",
                "    if len(logreg_valid_cols) > 0:\\n",
                "        pa_valid = np.stack([est.predict_proba(X)[:, 1] for est in logreg.estimators_], axis=1)\\n",
                "        pa[:, logreg_valid_cols] = pa_valid.astype(np.float32)\\n",
                "    pb = mlp_b.predict(X, batch_size=512, verbose=0)\\n",
                "    pc = mlp_c.predict(X, batch_size=512, verbose=0)\\n",
                "    return (pa + pb + pc) / 3.0\\n",
            ],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": None,
            "outputs": [],
            "source": [
                "sample_sub = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))\\n",
                "test_dir = Path(COMP_DIR) / 'test_soundscapes'\\n",
                "\\n",
                "pred_rows = []\\n",
                "for rid in sample_sub['row_id']:\\n",
                "    stem, end_s = rid.rsplit('_', 1)\\n",
                "    end_sec = int(end_s)\\n",
                "    start_sec = max(0, end_sec - int(CLIP_LOAD_SEC))\\n",
                "    fpath = test_dir / f'{stem}.ogg'\\n",
                "    y, _ = librosa.load(str(fpath), sr=SR_LOAD, mono=True, offset=start_sec, duration=CLIP_LOAD_SEC)\\n",
                "    n = int(CLIP_LOAD_SEC * SR_LOAD)\\n",
                "    if len(y) > n:\\n",
                "        y = y[:n]\\n",
                "    elif len(y) < n:\\n",
                "        y = np.pad(y, (0, n - len(y)))\\n",
                "    feat = extract_feat(y, SR_LOAD)[None, :]\\n",
                "    pred = predict_heads(feat)[0]\\n",
                "    row = {'row_id': rid}\\n",
                "    for c, p in zip(species_cols, pred):\\n",
                "        row[c] = float(np.clip(p, 0.0, 1.0))\\n",
                "    pred_rows.append(row)\\n",
                "\\n",
                "sub = pd.DataFrame(pred_rows)\\n",
                "sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')\\n",
                "sub[species_cols] = sub[species_cols].fillna(0.0).clip(0.0, 1.0)\\n",
                "sub.to_csv('/kaggle/working/submission.csv', index=False)\\n",
                "print('Saved /kaggle/working/submission.csv', sub.shape)"
            ],
        },
    ],
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python"},
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

kaggle_nb_path = BUNDLE_DIR / "kaggle_inference.ipynb"
with open(kaggle_nb_path, "w", encoding="utf-8") as f:
    json.dump(kaggle_nb, f, indent=2)

readme_path = BUNDLE_DIR / "README.txt"
readme_path.write_text(
    "BirdNET submission bundle (mix-trained).\\n"
    f"Created: {datetime.now().isoformat(timespec='seconds')}\\n"
    "Files:\\n"
    "- head_a_logreg.pkl\\n"
    "- head_b_mlp.keras\\n"
    "- head_c_mlp_mixup.keras\\n"
    "- metadata.npz\\n"
    "- kaggle_inference.ipynb\\n"
    "\\n"
    "Kaggle usage:\\n"
    "1) Upload this folder as a Kaggle Dataset.\\n"
    "2) Open kaggle_inference.ipynb and set MODEL_DIR.\\n"
    "3) Run all cells to create /kaggle/working/submission.csv.\\n",
    encoding="utf-8",
)

print("Exported bundle:", BUNDLE_DIR)
for p in sorted(BUNDLE_DIR.iterdir()):
    print(" -", p.name)

## Stage 6 — Ensemble (mean probability)

Rank averaging can be added later; start with a simple mean of calibrated-ish sigmoid outputs.

In [ ]:
pred_ens = (pred_a + pred_b + pred_c) / 3.0
auc_e, k3_e, npos = macro_auc_ge3(y_val, pred_ens)
print(f"Ensemble macro-AUC (≥3 pos): {auc_e:.5f} | species_kept≥3: {k3_e} | species_any_pos: {npos}")

pd.DataFrame(
    [
        {"head": "A_logreg", "macro_auc_ge3": auc_a},
        {"head": "B_mlp_weighted", "macro_auc_ge3": auc_b},
        {"head": "C_mlp_mixup", "macro_auc_ge3": auc_c},
        {"head": "mean_ABC", "macro_auc_ge3": auc_e},
    ]
)

## Notes

- **BirdNET downloads:** first `Analyzer()` init may fetch model weights (network required once).
- **RAM:** embedding ~35k clips is manageable; mixing multiplies rows — reduce `N_MIX_VIEWS` or `MAX_FOCAL` first.
- **Alignment:** focal labels are single-positive multi-hot; soundscape validation is true multi-label — macro-AUC subset ≥3 positives reduces noise from ultra-rare classes.
- To iterate faster, train heads on a subset of cached embeddings (`MAX_FOCAL`) before paying full embedding cost.
